# Practice Lab 3.4 - Function Calling and Tool Use (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 3 - Lesson 3.4 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-3.4.html` (6 exercises).
Each exercise below repeats the task and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- Exercises 1, 3, 4, 5, and 6 make Claude Sonnet 5 tool-use calls. **No key needed to run:** the
  keyless guard in the next cell mocks the client so every cell executes end to end with canned
  text. The mock answers directly instead of emitting a `tool_use` block, so the multi-turn tool
  loops finish in one round in keyless mode; set `ANTHROPIC_API_KEY` to reproduce the real
  multi-tool agent behaviour, the parallel speedup, and the eval accuracy numbers.
- Production context: Stripe 5-tool refund agent + Meridian Capital 3-tool underwriting; USD costs;
  synthetic payment / merchant / customer IDs only (`pay_TEST_`, `cust_`, `acct_TEST_`).
- Cost with a live key: well under the $40 lab cap (tool use routes to Sonnet 5).

In [ ]:
!pip install --quiet anthropic openai


In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------


---
## Exercise 1 - Define and call a single tool via the Anthropic SDK

**Task.** Define one tool `lookup_payment` that takes a `payment_id` string and returns a dict
with `status` and `amount_usd`. Wire it to a stub. Send the user query "What's the status of
pay_TEST_4421?" and execute the full round-trip: the model emits a `tool_use` block, your code
dispatches to the Python function, you send a `tool_result` block back, the model emits the final
text answer.

**Success criteria**
- `TOOLS` list has exactly one tool with all three keys (`name`, `description`, `input_schema`).
- Model emits a `tool_use` block on turn 1 (`r.stop_reason == "tool_use"`).
- You append the assistant turn AND the `tool_result` turn before the second call.
- Final text mentions both "captured" and the $amount.
- Total spend under $0.50 (2 Sonnet calls).

In [ ]:
# Worked solution - the smallest end-to-end tool round-trip the Anthropic SDK supports.
import anthropic
client = anthropic.Anthropic()

PAYMENTS = {
    "pay_TEST_4421": {"status": "captured", "amount_usd": 4500},
    "pay_TEST_9981": {"status": "refunded", "amount_usd": 1200},
}

def lookup_payment(payment_id: str) -> dict:
    return PAYMENTS.get(payment_id, {"status": "not_found", "amount_usd": 0})

TOOLS = [
    {
        "name": "lookup_payment",
        "description": "Returns the status and $amount of a Stripe payment by ID.",
        "input_schema": {
            "type": "object",
            "properties": {"payment_id": {"type": "string"}},
            "required": ["payment_id"],
        },
    },
]
assert set(TOOLS[0]) == {"name", "description", "input_schema"}   # all three keys present

messages = [{"role": "user", "content": "What's the status of pay_TEST_4421?"}]
r = client.messages.create(model="claude-sonnet-5", max_tokens=400,
                           tools=TOOLS, messages=messages)

# Turn 1: with a live key the model emits a tool_use block (r.stop_reason == "tool_use").
# The keyless mock answers directly, so guard the lookup and short-circuit to the text.
tool_use = next((b for b in r.content if getattr(b, "type", None) == "tool_use"), None)
if tool_use is None:
    print(r.content[0].text)   # keyless-mock path: canned text, no live tool round-trip
else:
    result = lookup_payment(**tool_use.input)
    messages.append({"role": "assistant", "content": r.content})            # assistant turn FIRST
    messages.append({"role": "user", "content": [                           # then the tool_result turn
        {"type": "tool_result", "tool_use_id": tool_use.id, "content": str(result)}
    ]})
    final = client.messages.create(model="claude-sonnet-5", max_tokens=400,
                                   tools=TOOLS, messages=messages)
    print(final.content[0].text)   # e.g. "Payment pay_TEST_4421 has been captured. Amount: $4,500."


---
## Exercise 2 - Rewrite a bad tool description (before / after)

**Task.** The `verify_kyc` tool ships with a one-line description and the agent skips it in 19 of
50 KYC-required scenarios. Rewrite the description with explicit WHEN to call / WHEN NOT to call /
PRE-REQUISITES / SIDE EFFECTS guidance, and add a GDPR `purpose` enum to `input_schema`. Show the
before and after as a single module so you can `diff` them.

**Success criteria**
- `TOOL_GOOD` description has all 4 named sections (WHEN / WHEN NOT / PRE-REQS / SIDE EFFECTS).
- WHEN section names the PRE-REQUISITE tool (`lookup_payment`).
- WHEN NOT section names at least one anti-pattern (already-called, not_found, etc.).
- `purpose` added to `input_schema` as a GDPR-required enum.
- The description does NOT exceed ~250 tokens (descriptions cost tokens on every call).

In [ ]:
# Worked solution - the one-line description rewritten with the 4-section template.

# --- BEFORE: the agent skips this 19/50 times ---
TOOL_BAD = {
    "name": "verify_kyc",
    "description": "Check KYC.",
    "input_schema": {
        "type": "object",
        "properties": {"customer_id": {"type": "string"}},
        "required": ["customer_id"],
    },
}

# --- AFTER: WHEN / WHEN NOT / PRE-REQUISITES / SIDE EFFECTS, plus a GDPR purpose enum ---
TOOL_GOOD = {
    "name": "verify_kyc",
    "description": """
WHEN TO CALL:
  - After lookup_payment confirms status=captured and the request is a refund.
  - Before process_refund - process_refund will reject without a fresh KYC.

WHEN NOT TO CALL:
  - If lookup_payment returned status=not_found.
  - If you have already called verify_kyc for this customer_id in this conversation
    (the result is stable within the session).

PRE-REQUISITES:
  - Must have a customer_id (returned by lookup_payment.customer_id).
  - The customer_id must be a Stripe-prefixed ID, not an SSN or passport.

SIDE EFFECTS:
  - None. Read-only. Logs a GDPR audit record with the declared purpose.

Returns: {"kyc_state": "verified"|"expired"|"missing", "last_verified_at": ISO8601}
""".strip(),
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {"type": "string", "description": "Stripe cust_ prefix"},
            "purpose": {
                "type": "string",
                "enum": ["refund_check", "fraud_review", "onboarding"],
                "description": "GDPR-required purpose for PII access",
            },
        },
        "required": ["customer_id", "purpose"],
    },
}

# The 5 hard-mode queries the BAD description skips (~3/5) - the AFTER description recovers them.
SMOKE = [
    "Refund pay_TEST_4421 - customer cust_991 wants their money back",
    "Customer cust_991 is asking for their refund status",
    "Process refund of $1200 for pay_TEST_9981",
    "Refund please for cust_8821",
    "Need to refund the $4500 payment for cust_991",
]

# Self-checks against the success criteria:
d = TOOL_GOOD["description"]
for section in ("WHEN TO CALL", "WHEN NOT TO CALL", "PRE-REQUISITES", "SIDE EFFECTS"):
    assert section in d, f"missing section: {section}"
assert "lookup_payment" in d                                   # WHEN names the pre-req tool
assert "not_found" in d or "already" in d                      # WHEN NOT names an anti-pattern
assert TOOL_GOOD["input_schema"]["properties"]["purpose"]["enum"]   # GDPR purpose enum present
approx_tokens = len(d) // 4                                     # ~4 chars/token rule of thumb
assert approx_tokens < 250, "description is over the ~250-token budget"
print(d)
print(f"\nOK - 4 sections, pre-req tool named, purpose enum added (~{approx_tokens} tokens).")


---
## Exercise 3 - Two-tool multi-turn loop (proper tool_use + tool_result handling)

**Task.** Implement a 2-tool loop with `lookup_payment` + `verify_kyc`. User query: "Can I refund
pay_TEST_4421?" The agent must call `lookup_payment`, then call `verify_kyc` with the customer_id
from the first result, then emit a final text decision. Use a generic `run_tool_loop` so Exercises
4-6 can reuse it.

**Success criteria**
- Loop runs to `end_turn` in <= 4 rounds on the happy path.
- Both tools called exactly once (verify with a counter inside the stubs).
- Assistant turn appended BEFORE the `tool_result` user turn (no API 400).
- `tool_use_id` in each result matches the corresponding `tool_use.id`.
- Final text names APPROVE or REJECT (no hedging).
- Total spend under $1 (3-4 Sonnet calls).

In [ ]:
# Worked solution - the canonical multi-turn tool loop (copy-paste into Ex 4-6).
import anthropic
client = anthropic.Anthropic()

PAYMENTS = {"pay_TEST_4421": {"status": "captured", "amount_usd": 4500, "customer_id": "cust_991"}}
KYC = {"cust_991": {"kyc_state": "verified", "last_verified_at": "2026-04-12T08:00:00Z"}}

def lookup_payment(payment_id: str) -> dict:
    return PAYMENTS.get(payment_id, {"status": "not_found"})

def verify_kyc(customer_id: str, purpose: str = "refund_check") -> dict:
    return KYC.get(customer_id, {"kyc_state": "missing"})

TOOL_FNS = {"lookup_payment": lookup_payment, "verify_kyc": verify_kyc}

TOOLS = [
    {"name": "lookup_payment",
     "description": "Returns status, amount_usd, customer_id for a Stripe payment. Call FIRST.",
     "input_schema": {"type": "object", "properties": {"payment_id": {"type": "string"}},
                      "required": ["payment_id"]}},
    {"name": "verify_kyc",
     "description": "Returns kyc_state for a customer. Call AFTER lookup_payment, BEFORE any refund "
                    "decision. Requires a GDPR purpose.",
     "input_schema": {"type": "object", "properties": {
         "customer_id": {"type": "string"},
         "purpose": {"type": "string", "enum": ["refund_check", "fraud_review", "onboarding"]}},
       "required": ["customer_id", "purpose"]}},
]

SYS = """You are a Stripe refund agent.
Steps: 1) lookup_payment, 2) verify_kyc, 3) decide APPROVE or REJECT in one sentence."""

def run_tool_loop(query: str, max_rounds: int = 5) -> dict:
    messages = [{"role": "user", "content": query}]
    for i in range(max_rounds):
        r = client.messages.create(model="claude-sonnet-5", max_tokens=500,
                                   system=SYS, tools=TOOLS, messages=messages)
        if r.stop_reason == "end_turn":
            txt = next((b.text for b in r.content if getattr(b, "type", None) == "text"), "")
            return {"status": "OK", "rounds": i + 1, "final": txt}
        # tool_use path: append the assistant turn FIRST, then run every tool_use block
        messages.append({"role": "assistant", "content": r.content})
        tool_results = []
        for b in r.content:
            if getattr(b, "type", None) == "tool_use":
                fn = TOOL_FNS.get(b.name)
                out = fn(**b.input) if fn else {"error": "unknown_tool"}
                tool_results.append({"type": "tool_result", "tool_use_id": b.id, "content": str(out)})
        messages.append({"role": "user", "content": tool_results})   # one user turn, N tool_results
    return {"status": "CAP_HIT", "rounds": max_rounds, "final": "agent did not converge"}

out = run_tool_loop("Can I refund pay_TEST_4421?")
print(f"[{out['status']}] rounds={out['rounds']} :: {out['final']}")
# Live key -> "[OK] rounds=3 :: APPROVE - pay_TEST_4421 is captured ($4,500), cust_991 KYC verified."


---
## Exercise 4 - Convert a sequential 3-tool flow to parallel (measure the latency drop)

**Task.** Build the Meridian Capital underwriting agent with 3 lookup tools (`lookup_credit`,
`lookup_tax`, `lookup_bank_statements`). Run it twice: once with sequential descriptions, once
with parallel ("INDEPENDENT - can be called in parallel") descriptions. When the model emits all 3
`tool_use` blocks in one turn, execute them concurrently with `asyncio.gather` and return all 3
`tool_result` blocks in one user turn. Measure wall-clock latency for each and print a 2-row
comparison.

**Success criteria**
- 2 rows printed: sequential and parallel.
- Speedup >= 1.8x (1.5x and below means the model did not actually parallelize).
- On the parallel run, the assistant turn carries >= 2 `tool_use` blocks in one turn.
- Synthetic SSN / tax_id / account_id only - no real PII.
- Total spend under $2 (2 short Sonnet runs).

In [ ]:
# Worked solution - parallel tool fan-out with asyncio.gather + a latency comparison.
import asyncio, time, anthropic
client = anthropic.Anthropic()

async def lookup_credit(ssn: str) -> dict:
    await asyncio.sleep(0.4)                     # simulate network latency
    return {"score": 782, "ssn": ssn}

async def lookup_tax(tax_id: str) -> dict:
    await asyncio.sleep(0.4)
    return {"annual_revenue_usd": 15_000, "tax_id": tax_id}

async def lookup_bank_statements(account_id: str) -> dict:
    await asyncio.sleep(0.4)
    return {"avg_monthly_balance_usd": 1_000, "account_id": account_id}

TOOL_FNS = {"lookup_credit": lookup_credit, "lookup_tax": lookup_tax,
            "lookup_bank_statements": lookup_bank_statements}

def make_tools(parallel: bool) -> list:
    indep = ("INDEPENDENT. Can be called in parallel with the other lookups." if parallel
             else "Call in sequence; wait for each to return before the next.")
    return [
        {"name": "lookup_credit",
         "description": f"Returns credit score for an SSN. {indep}",
         "input_schema": {"type": "object", "properties": {"ssn": {"type": "string"}}, "required": ["ssn"]}},
        {"name": "lookup_tax",
         "description": f"Returns sales-tax annual revenue for a tax ID. {indep}",
         "input_schema": {"type": "object", "properties": {"tax_id": {"type": "string"}}, "required": ["tax_id"]}},
        {"name": "lookup_bank_statements",
         "description": f"Returns avg balance for an account. {indep}",
         "input_schema": {"type": "object", "properties": {"account_id": {"type": "string"}}, "required": ["account_id"]}},
    ]

SYS = ("You are a Meridian Capital underwriting agent. Gather credit, sales-tax, and "
       "bank-statement data, then summarize.")

async def run_once(parallel: bool) -> float:
    tools = make_tools(parallel)
    messages = [{"role": "user", "content": (
        "Underwrite: SSN=123456789 tax_id=TAX-1234567 account=acct_TEST_77. "
        "Gather the 3 inputs and summarize.")}]
    t0 = time.perf_counter()
    for _ in range(5):
        r = client.messages.create(model="claude-sonnet-5", max_tokens=600,
                                   system=SYS, tools=tools, messages=messages)
        if r.stop_reason == "end_turn":
            break
        messages.append({"role": "assistant", "content": r.content})
        # Execute ALL tool_use blocks from this turn concurrently:
        calls = [(b.name, b.input, b.id) for b in r.content if getattr(b, "type", None) == "tool_use"]
        results = await asyncio.gather(*[TOOL_FNS[n](**a) for n, a, _ in calls]) if calls else []
        messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": cid, "content": str(res)}
            for (_, _, cid), res in zip(calls, results)]})
    return time.perf_counter() - t0

async def main():
    lat_seq = await run_once(parallel=False)
    lat_par = await run_once(parallel=True)
    print(f"{'mode':10s} | wall-clock s")
    print("-" * 28)
    print(f"{'sequential':10s} | {lat_seq:.2f}")
    print(f"{'parallel':10s} | {lat_par:.2f}")
    print(f"\nSpeedup: {lat_seq / max(lat_par, 1e-6):.2f}x")

# In a notebook (Colab/Jupyter) the event loop is already running, so use top-level await
# instead of asyncio.run(main()). With a live key you should see ~2-3x on the parallel row.
await main()


---
## Exercise 5 - The 5-tool Stripe refund agent (cap + idempotency)

**Task.** Implement all 5 tools (`lookup_payment`, `check_refund_window`, `verify_kyc`,
`process_refund`, `escalate_to_human`) with the 4-section description template. Add a 5-round cap.
Derive `idempotency_key = sha256(payment_id + reason)[:16]` deterministically before each
`process_refund`. Run the 3 lesson scenarios (happy refund, expired window, missing KYC) and
assert: no double-refund, no infinite loop, correct decision per scenario.

**Success criteria**
- All 5 tools wired, each with the 4-section description template.
- 3 scenarios run; only the happy path executes `process_refund` (ledger len == 1).
- Window-expired and KYC-missing both escalate (no refund processed).
- `idempotency_key` is deterministic - re-running the happy scenario adds 0 new ledger entries.
- `MAX_ROUNDS = 5` cap present; a forced cap-hit test fires the fallback escalation.
- `check_refund_window` and `verify_kyc` fire in the SAME assistant turn at least once.
- Total spend under $5 (10-15 Sonnet calls).

In [ ]:
# Worked solution - the lesson's canonical 5-tool reference design, assembled and runnable.
import hashlib, anthropic
client = anthropic.Anthropic()

# --- Tool stubs (production: real Stripe endpoints) ---
PAYMENTS = {
    "pay_TEST_OK":   {"status": "captured", "amount_usd": 4500, "customer_id": "cust_OK",   "captured_at": "2026-05-10"},
    "pay_TEST_OLD":  {"status": "captured", "amount_usd": 1200, "customer_id": "cust_OK",   "captured_at": "2025-01-01"},
    "pay_TEST_NOKY": {"status": "captured", "amount_usd": 900,  "customer_id": "cust_NOKY", "captured_at": "2026-05-12"},
}
KYC = {"cust_OK": {"kyc_state": "verified"}, "cust_NOKY": {"kyc_state": "missing"}}
REFUND_LEDGER = {}   # idempotency_key -> result (the production idempotency cache)

def lookup_payment(payment_id, purpose="refund_check"):
    return PAYMENTS.get(payment_id, {"status": "not_found"})

def check_refund_window(payment_id):
    p = PAYMENTS.get(payment_id, {})
    return {"within_window": p.get("captured_at", "2020-01-01") >= "2026-02-20"}

def verify_kyc(customer_id, purpose="refund_check"):
    return KYC.get(customer_id, {"kyc_state": "missing"})

def process_refund(payment_id, amount_usd, reason, idempotency_key):
    if idempotency_key in REFUND_LEDGER:
        return REFUND_LEDGER[idempotency_key]                 # idempotent replay - no double money
    out = {"refund_id": f"rfnd_{idempotency_key[:8]}", "status": "processed", "amount_usd": amount_usd}
    REFUND_LEDGER[idempotency_key] = out
    return out

def escalate_to_human(reason):
    return {"ticket_id": f"ST-REF-{abs(hash(reason)) % 10000:04d}", "status": "open"}

TOOL_FNS = {"lookup_payment": lookup_payment, "check_refund_window": check_refund_window,
            "verify_kyc": verify_kyc, "process_refund": process_refund,
            "escalate_to_human": escalate_to_human}

def desc(when, when_not, prereqs, side):
    return f"WHEN: {when}\nWHEN NOT: {when_not}\nPRE-REQS: {prereqs}\nSIDE EFFECTS: {side}"

TOOLS = [
    {"name": "lookup_payment",
     "description": desc("FIRST call. Always.", "Never skip.", "None.", "None (read-only)."),
     "input_schema": {"type": "object", "properties": {"payment_id": {"type": "string"}}, "required": ["payment_id"]}},
    {"name": "check_refund_window",
     "description": desc("After lookup_payment confirms status=captured.", "If status != captured.",
                         "lookup_payment must have returned captured.",
                         "None. INDEPENDENT - parallel-safe with verify_kyc."),
     "input_schema": {"type": "object", "properties": {"payment_id": {"type": "string"}}, "required": ["payment_id"]}},
    {"name": "verify_kyc",
     "description": desc("After lookup_payment. INDEPENDENT - parallel-safe with check_refund_window.",
                         "If you already called this for this customer_id in-session.",
                         "customer_id from lookup_payment.customer_id.", "None. Logs GDPR purpose."),
     "input_schema": {"type": "object", "properties": {
         "customer_id": {"type": "string"},
         "purpose": {"type": "string", "enum": ["refund_check", "fraud_review", "onboarding"]}},
       "required": ["customer_id", "purpose"]}},
    {"name": "process_refund",
     "description": desc("ONLY after lookup_payment, check_refund_window=true, verify_kyc=verified.",
                         "Any pre-req failing. Never call twice for the same payment_id.",
                         "All three reads must have succeeded.",
                         "Moves real $. Pass deterministic idempotency_key = sha256(payment_id+reason)[:16]."),
     "input_schema": {"type": "object", "properties": {
         "payment_id": {"type": "string"}, "amount_usd": {"type": "integer"},
         "reason": {"type": "string"},     "idempotency_key": {"type": "string"}},
       "required": ["payment_id", "amount_usd", "reason", "idempotency_key"]}},
    {"name": "escalate_to_human",
     "description": desc("Any failed pre-req (window expired, KYC missing, payment not_found).",
                         "If process_refund succeeded.", "None.", "Opens a ticket."),
     "input_schema": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]}},
]

SYS = """You are a Stripe refund agent. Follow the tool descriptions strictly.
For process_refund, derive idempotency_key = sha256(payment_id + reason)[:16] before calling."""

MAX_ROUNDS = 5

def run_agent(query: str) -> dict:
    messages = [{"role": "user", "content": query}]
    for i in range(MAX_ROUNDS):
        r = client.messages.create(model="claude-sonnet-5", max_tokens=700,
                                   system=SYS, tools=TOOLS, messages=messages)
        if r.stop_reason == "end_turn":
            txt = next((b.text for b in r.content if getattr(b, "type", None) == "text"), "")
            return {"status": "OK", "rounds": i + 1, "final": txt}
        messages.append({"role": "assistant", "content": r.content})
        results = []
        for b in r.content:
            if getattr(b, "type", None) == "tool_use":
                out = TOOL_FNS[b.name](**b.input)
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": str(out)})
        messages.append({"role": "user", "content": results})
    fb = escalate_to_human(f"agent cap-hit on: {query[:40]}")     # cap-hit fallback
    return {"status": "CAP_HIT", "rounds": MAX_ROUNDS, "final": f"ESCALATED {fb['ticket_id']}"}

for s in ["Refund pay_TEST_OK $4500, customer wants money back",
          "Refund pay_TEST_OLD $1200 - 14 months old",
          "Refund pay_TEST_NOKY $900 urgent"]:
    out = run_agent(s)
    print(f"[{out['status']:7s}] rounds={out['rounds']} :: {out['final'][:90]}")
print(f"\nRefund ledger ({len(REFUND_LEDGER)} entries): {list(REFUND_LEDGER.keys())}")
# Live key: only the happy path processes a refund, so the ledger holds exactly 1 entry.


---
## Exercise 6 (GRADED) - production refund agent + tools-spec + 50-query eval

**Task.** Ship four files in `solutions/practice-3.4/ex6/`:
1. `refund_agent.py` - the 5-tool agent productionized: parallel fan-out on the 3 reads,
   deterministic idempotency, 5-round cap, structured logging per round (round, tool, purpose,
   latency_ms, $cost), and a `dry_run=True` mode that short-circuits `process_refund` with a
   logged intent.
2. `tools_spec.md` - one section per tool: WHEN / WHEN NOT / PRE-REQUISITES / SIDE EFFECTS / GDPR
   purpose / idempotency contract.
3. `eval_50.jsonl` + `eval.py` - 50 refund scenarios labelled with expected outcome; `eval.py`
   runs the agent against all 50 and prints accuracy (Wilson 95% CI), mean rounds, P95 latency,
   $total, and the count of double-process attempts caught by the ledger.
4. `eval_report.md` - 250 words: headline accuracy + CI, one tool-description finding, and 3
   prod-gap items (PII redaction in logs, a real idempotency store, a retry policy).

**Bonus (+1)** - `dry_run=True` on `process_refund` returns a synthetic `refund_id` with
`status: "DRY_RUN"`, logs the intent, but does NOT mutate the ledger (the mode customer compliance
teams use during UAT).

The worked solution below productionizes the Exercise 5 agent (`dry_run` + structured log + cost),
writes a 50-scenario `eval_50.jsonl` to disk, and runs the eval harness end to end. In KEYLESS
MOCK mode the agent returns canned text so accuracy prints ~0%; set `ANTHROPIC_API_KEY` for the
real numbers (target overall >= 90%). `tools_spec.md` and `eval_report.md` are prose deliverables -
their required contents are in the grading rubric cell at the end.

In [ ]:
# Worked solution (Exercise 6) - the production package as one runnable cell.
import hashlib, time, json, math, anthropic
client = anthropic.Anthropic()

RATE_IN, RATE_OUT = 2 / 1_000_000, 10 / 1_000_000   # $ per Sonnet 5 token (Aug 2026: $2 in / $10 out per 1M)

# --- synthetic Stripe-style stores (production: real Stripe endpoints) ---
PAYMENTS = {
    "pay_TEST_OK":   {"status": "captured", "amount_usd": 4500, "customer_id": "cust_OK",   "captured_at": "2026-05-10"},
    "pay_TEST_OLD":  {"status": "captured", "amount_usd": 1200, "customer_id": "cust_OK",   "captured_at": "2025-01-01"},
    "pay_TEST_NOKY": {"status": "captured", "amount_usd": 900,  "customer_id": "cust_NOKY", "captured_at": "2026-05-12"},
    "pay_TEST_GONE": {"status": "not_found"},
    "pay_TEST_DONE": {"status": "refunded", "amount_usd": 700, "customer_id": "cust_OK", "captured_at": "2026-05-01"},
}
KYC = {"cust_OK": {"kyc_state": "verified"}, "cust_NOKY": {"kyc_state": "missing"}}
REFUND_LEDGER = {}      # idempotency_key -> result
STRUCTURED_LOG = []     # one entry per tool call per round

def idem_key(payment_id: str, reason: str) -> str:
    return hashlib.sha256(f"{payment_id}|{reason}".encode()).hexdigest()[:16]

def lookup_payment(payment_id, purpose="refund_check"):
    return PAYMENTS.get(payment_id, {"status": "not_found"})

def check_refund_window(payment_id):
    p = PAYMENTS.get(payment_id, {})
    return {"within_window": p.get("captured_at", "2020-01-01") >= "2026-02-20"}

def verify_kyc(customer_id, purpose="refund_check"):
    return KYC.get(customer_id, {"kyc_state": "missing"})

def process_refund(payment_id, amount_usd, reason, idempotency_key, dry_run=False):
    if idempotency_key in REFUND_LEDGER:
        return REFUND_LEDGER[idempotency_key]                          # idempotent replay
    out = {"refund_id": f"rfnd_{idempotency_key[:8]}", "amount_usd": amount_usd,
           "status": "DRY_RUN" if dry_run else "processed"}
    if not dry_run:
        REFUND_LEDGER[idempotency_key] = out                           # dry-run never mutates
    return out

def escalate_to_human(reason):
    return {"ticket_id": f"ST-REF-{abs(hash(reason)) % 10000:04d}", "status": "open"}

TOOL_FNS = {"lookup_payment": lookup_payment, "check_refund_window": check_refund_window,
            "verify_kyc": verify_kyc, "process_refund": process_refund,
            "escalate_to_human": escalate_to_human}

def desc(when, when_not, prereqs, side):
    return f"WHEN: {when}\nWHEN NOT: {when_not}\nPRE-REQS: {prereqs}\nSIDE EFFECTS: {side}"

TOOLS = [
    {"name": "lookup_payment",
     "description": desc("FIRST call. Always.", "Never skip.", "None.", "None (read-only)."),
     "input_schema": {"type": "object", "properties": {
         "payment_id": {"type": "string"},
         "purpose": {"type": "string", "enum": ["refund_check", "fraud_review", "onboarding"]}},
       "required": ["payment_id"]}},
    {"name": "check_refund_window",
     "description": desc("After lookup_payment confirms status=captured.", "If status != captured.",
                         "lookup_payment must have returned captured.",
                         "None. INDEPENDENT - parallel-safe with verify_kyc."),
     "input_schema": {"type": "object", "properties": {"payment_id": {"type": "string"}}, "required": ["payment_id"]}},
    {"name": "verify_kyc",
     "description": desc("After lookup_payment. INDEPENDENT - parallel-safe with check_refund_window.",
                         "If you already called this for this customer_id in-session.",
                         "customer_id from lookup_payment.customer_id.", "None. Logs GDPR purpose."),
     "input_schema": {"type": "object", "properties": {
         "customer_id": {"type": "string"},
         "purpose": {"type": "string", "enum": ["refund_check", "fraud_review", "onboarding"]}},
       "required": ["customer_id", "purpose"]}},
    {"name": "process_refund",
     "description": desc("ONLY after lookup_payment, check_refund_window=true, verify_kyc=verified.",
                         "Any pre-req failing. Never call twice for the same payment_id.",
                         "All three reads must have succeeded.",
                         "Moves real $. idempotency_key = sha256(payment_id+reason)[:16]."),
     "input_schema": {"type": "object", "properties": {
         "payment_id": {"type": "string"}, "amount_usd": {"type": "integer"},
         "reason": {"type": "string"},     "idempotency_key": {"type": "string"}},
       "required": ["payment_id", "amount_usd", "reason", "idempotency_key"]}},
    {"name": "escalate_to_human",
     "description": desc("Any failed pre-req (window expired, KYC missing, payment not_found).",
                         "If process_refund succeeded.", "None.", "Opens a ticket."),
     "input_schema": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]}},
]

SYS = """You are a Stripe refund agent. Follow the tool descriptions strictly.
For process_refund, derive idempotency_key = sha256(payment_id + reason)[:16] before calling."""

def run_agent(query: str, dry_run: bool = False, max_rounds: int = 5) -> dict:
    messages = [{"role": "user", "content": query}]
    t0 = time.perf_counter(); total_cost = 0.0
    for i in range(max_rounds):
        r = client.messages.create(model="claude-sonnet-5", max_tokens=800,
                                   system=SYS, tools=TOOLS, messages=messages)
        total_cost += r.usage.input_tokens * RATE_IN + r.usage.output_tokens * RATE_OUT
        if r.stop_reason == "end_turn":
            txt = next((b.text for b in r.content if getattr(b, "type", None) == "text"), "")
            return {"status": "OK", "rounds": i + 1, "final": txt,
                    "latency_s": time.perf_counter() - t0, "cost_usd": total_cost}
        messages.append({"role": "assistant", "content": r.content})
        results = []
        for b in r.content:
            if getattr(b, "type", None) == "tool_use":
                t1 = time.perf_counter()
                args = dict(b.input)
                if b.name == "process_refund":                          # derive the key in Python,
                    args["idempotency_key"] = idem_key(args["payment_id"], args.get("reason", "refund"))
                    args["dry_run"] = dry_run                            # not in the model's hands
                out = TOOL_FNS[b.name](**args)
                STRUCTURED_LOG.append({"round": i + 1, "tool": b.name,
                                       "purpose": b.input.get("purpose"),
                                       "latency_ms": int((time.perf_counter() - t1) * 1000)})
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": str(out)})
        messages.append({"role": "user", "content": results})
    fb = escalate_to_human(f"cap-hit: {query[:40]}")
    return {"status": "CAP_HIT", "rounds": max_rounds, "final": f"ESCALATED {fb['ticket_id']}",
            "latency_s": time.perf_counter() - t0, "cost_usd": total_cost}

# --- eval_50.jsonl: 50 labelled scenarios (12 x 4 outcomes + 2 edge cases) ---
TEMPLATES = [
    ("Refund pay_TEST_OK ${amt}, customer wants money back",      "refund"),
    ("Refund pay_TEST_OLD ${amt} - the charge is 14 months old",  "escalate"),
    ("Refund pay_TEST_NOKY ${amt} urgent, customer waiting",      "escalate"),
    ("Refund pay_TEST_GONE ${amt} please",                        "escalate"),
]
GOLDEN = []
for tmpl, expected in TEMPLATES:
    for j in range(12):
        GOLDEN.append({"query": tmpl.replace("${amt}", str(100 + j * 10)), "expected": expected})
GOLDEN.append({"query": "Refund pay_TEST_OK $4500 again, customer double-asked", "expected": "refund"})   # replay
GOLDEN.append({"query": "Refund pay_TEST_DONE $700, already refunded once",       "expected": "already"})  # edge
with open("eval_50.jsonl", "w", encoding="utf-8") as f:
    for ex in GOLDEN:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

def wilson(hits: int, n: int, z: float = 1.96) -> tuple:
    if n == 0:
        return (0.0, 0.0)
    p = hits / n; denom = 1 + z * z / n
    c = (p + z * z / (2 * n)) / denom
    s = z * math.sqrt((p * (1 - p) + z * z / (4 * n)) / n) / denom
    return c - s, c + s

queries = [json.loads(l) for l in open("eval_50.jsonl", encoding="utf-8")]
runs = [run_agent(q["query"]) for q in queries]
hits = sum(1 for q, r in zip(queries, runs) if q["expected"] in r["final"].lower())
lo, hi = wilson(hits, len(runs))
lats = sorted(r["latency_s"] for r in runs)
p95 = lats[min(int(0.95 * len(lats)), len(lats) - 1)]
print(f"eval set: {len(queries)} scenarios (12 x 4 outcomes + 2 edge cases)")
print(f"accuracy: {hits / len(runs) * 100:.1f}% [95% CI {lo * 100:.1f}-{hi * 100:.1f}]")
print(f"mean rounds: {sum(r['rounds'] for r in runs) / len(runs):.2f}")
print(f"P95 latency: {p95:.3f}s | total cost: ${sum(r['cost_usd'] for r in runs):.4f}")
print(f"unique refunds in ledger (double-process attempts collapsed): {len(REFUND_LEDGER)}")
print("\n(KEYLESS MOCK -> canned text, so accuracy prints ~0%. Set ANTHROPIC_API_KEY for real numbers: target >= 90%.)")


---
## Exercise 6 - grading rubric + submission checklist

**Grading rubric (10 points, +1 bonus)**

| Axis | Weight | Full marks |
|---|---|---|
| Tool design | 3 | All 5 tools defined with clean input schemas and correct JSON Schema types. PII-touching tools (`verify_kyc`, `lookup_payment`) carry a `purpose` enum for GDPR. `process_refund` lists `idempotency_key` in `required`. One tool, one job. |
| Description quality | 3 | Each of the 5 descriptions has the 4 named sections (WHEN / WHEN NOT / PRE-REQS / SIDE EFFECTS). `process_refund` names the exact idempotency_key derivation. `check_refund_window` + `verify_kyc` are marked INDEPENDENT to enable parallel fan-out. No description exceeds ~250 tokens. |
| Parallel + dependency handling | 2 | At least one assistant turn in the eval emits >= 2 `tool_use` blocks. Sequential edges (lookup_payment -> everything; the 3 reads -> process_refund) are respected. Verified via the structured log. |
| Idempotency + caps | 2 | Deterministic `idempotency_key` derived in Python (not invented by the model). `REFUND_LEDGER` short-circuits replays. `MAX_ROUNDS = 5` with a tested fallback escalation on cap-hit. The duplicate-refund query is caught. |
| Bonus: dry-run mode | +1 | `dry_run=True` returns a synthetic `refund_id` with `status: "DRY_RUN"`, logs the intent, does NOT mutate the ledger. Documented in `tools_spec.md` with the UAT / compliance rationale. |

**Submission checklist**
- [ ] `refund_agent.py` runs end to end on the 50-query eval without manual intervention.
- [ ] `tools_spec.md` has one section per tool with all 4 description segments + GDPR purpose + idempotency contract.
- [ ] `eval_50.jsonl` has 50 labelled scenarios across the 4 outcomes + duplicate-refund + already-refunded edge cases.
- [ ] `eval_report.md` is 250 words; accuracy with Wilson 95% CI; one description-engineering finding; 3 prod-gap items (PII redaction, real idempotency store, retry policy).
- [ ] Structured log includes the `purpose` field on every PII-touching tool call (GDPR audit trail).
- [ ] Synthetic IDs only (`pay_TEST_`, `cust_`, `acct_TEST_`) - no real Stripe payment IDs or PANs.
- [ ] Total spend under $25 (50 multi-round Sonnet runs; the $40 lab cap absorbs Exercises 1-5 + this).

**Feedback SLA** - submit in the cohort Discord `#practice-labs` channel; rubric-graded feedback is
returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked notebook
and the rubric above to self-assess right away.